In [ ]:
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/project")


def read_env(path):
    values = {}
    for number, line in enumerate(path.read_text().splitlines(), 1):
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        key, separator, value = line.partition("=")
        if not separator or not key or key != key.strip():
            raise ValueError(f"Invalid KEY=value line {number} in {path}")
        values[key] = value
    return values


CONFIG = read_env(PROJECT_DIR / ".colab.env")
R2_CREDENTIALS = read_env(Path("/content/.colab-r2.env"))
for key in ("R2_BUCKET", "R2_ARTIFACT_PREFIX", "EXPECT_GPU"):
    if not CONFIG.get(key):
        raise ValueError(f"Missing {key} in .colab.env")
for key in ("R2_ACCOUNT_ID", "R2_ACCESS_KEY_ID", "R2_SECRET_ACCESS_KEY"):
    if not R2_CREDENTIALS.get(key):
        raise ValueError(f"Missing {key} in the external R2 credentials file")
if CONFIG["EXPECT_GPU"] not in ("true", "false"):
    raise ValueError("EXPECT_GPU must be true or false")
ARTIFACT_PREFIX = CONFIG["R2_ARTIFACT_PREFIX"].strip("/")
if not ARTIFACT_PREFIX:
    raise ValueError("R2_ARTIFACT_PREFIX must name a folder")
if CONFIG.get("R2_DATA_PREFIX", "").strip("/") != ARTIFACT_PREFIX:
    raise ValueError("R2_DATA_PREFIX must match R2_ARTIFACT_PREFIX")

DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "output"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)

gpu_command = shutil.which("nvidia-smi")
gpu = (
    subprocess.run([gpu_command, "-L"], capture_output=True, text=True)
    if gpu_command
    else None
)
if CONFIG["EXPECT_GPU"] == "true" and (gpu is None or gpu.returncode != 0):
    raise RuntimeError("GPU expected but nvidia-smi did not find one")
print(f"Python: {sys.version.split()[0]} on {platform.platform()}")
print(f"Working directory: {Path.cwd()}")
print(f"GPU: {gpu.stdout.strip() if gpu and gpu.returncode == 0 else 'none'}")
print(f"R2 bucket: {CONFIG['R2_BUCKET']} | data: {DATA_DIR} | output: {OUTPUT_DIR}")
print(f"Artifact prefix: {ARTIFACT_PREFIX}")
print("R2 credentials: present")

In [ ]:
%pip install -qq --disable-pip-version-check uv boto3

import subprocess

requirements = PROJECT_DIR / "requirements-colab.txt"
subprocess.run(
    [
        "uv",
        "export",
        "--frozen",
        "--no-dev",
        "--extra",
        "experiment",
        "--prune",
        "torch",
        "--prune",
        "numpy",
        "--prune",
        "fsspec",
        "--prune",
        "rich",
        "--prune",
        "colorama",
        "--no-emit-project",
        "--no-hashes",
        "--format",
        "requirements.txt",
        "--output-file",
        str(requirements),
        "--project",
        str(PROJECT_DIR),
    ],
    check=True,
)
%pip install -qq --disable-pip-version-check --requirement {requirements}
%pip install -qq --disable-pip-version-check --no-deps {PROJECT_DIR}

In [ ]:
from jlens_reasoning.environments.colab import source_bundle_sha256

PROJECT_SOURCE_SHA256 = source_bundle_sha256(PROJECT_DIR)
print(f"Project source SHA-256: {PROJECT_SOURCE_SHA256}")

In [ ]:
def r2_client():
    import boto3

    return boto3.client(
        "s3",
        endpoint_url=f"https://{R2_CREDENTIALS['R2_ACCOUNT_ID']}.r2.cloudflarestorage.com",
        aws_access_key_id=R2_CREDENTIALS["R2_ACCESS_KEY_ID"],
        aws_secret_access_key=R2_CREDENTIALS["R2_SECRET_ACCESS_KEY"],
        region_name="auto",
    )


def upload_artifacts():
    """Upload files in OUTPUT_DIR under the project's artifact prefix."""
    client = r2_client()
    count = 0
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_symlink():
            raise ValueError(f"Refusing to upload symlink: {path}")
        if path.is_file():
            key = f"{ARTIFACT_PREFIX}/{path.relative_to(OUTPUT_DIR).as_posix()}"
            client.upload_file(str(path), CONFIG["R2_BUCKET"], key)
            count += 1
    print(f"Uploaded {count} file(s) from {OUTPUT_DIR}")

In [ ]:
from jlens_reasoning.environments.colab import download_r2_inputs

download_r2_inputs(
    client=r2_client(),
    bucket=CONFIG["R2_BUCKET"],
    prefix=ARTIFACT_PREFIX,
    destination=DATA_DIR,
    paths=("runs/flenqa-full-run/model_outputs.parquet",),
)

In [ ]:
from jlens_reasoning.environments.colab import initialize_colab

context = initialize_colab(enable_wandb=False, require_cuda=True)
context

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from jlens_reasoning.evaluation import evaluate_paper_binary

RESULT_DIR = OUTPUT_DIR / "runs/flenqa-accuracy"
RESULT_PATH = RESULT_DIR / "results.parquet"
MODEL_OUTPUT_PATH = context.runs_dir / "flenqa-full-run" / "model_outputs.parquet"
LENGTHS = (250, 500, 1000, 2000, 3000)
EXPECTED_UNIQUE_COUNTS = {
    250: 300,
    500: 2_368,
    1000: 2_394,
    2000: 2_400,
    3000: 2_400,
}

In [ ]:
model_outputs = pq.read_table(MODEL_OUTPUT_PATH)
assert model_outputs.num_rows == 9_862
assert "generated_text" in model_outputs.column_names
model_outputs.schema

In [ ]:
records = model_outputs.to_pylist()
verdicts = []
correctness = []
for record in records:
    evaluation = evaluate_paper_binary(
        record["generated_text"], expected=record["label"]
    )
    verdicts.append(evaluation.verdict)
    correctness.append(evaluation.correct)

actual_counts = Counter(record["ctx_size"] for record in records)
assert dict(actual_counts) == EXPECTED_UNIQUE_COUNTS
paper_counts = Counter()
for record in records:
    paper_counts[record["ctx_size"]] += record["paper_weight"]
assert dict(paper_counts) == {length: 600 for length in LENGTHS}
results = model_outputs.append_column(
    "verdict", pa.array(verdicts, type=pa.bool_())
).append_column("correct", pa.array(correctness, type=pa.bool_()))
{"rows": results.num_rows, "counts_by_length": dict(actual_counts)}

In [ ]:
assert results.num_rows == 9_862
assert RESULT_PATH.name == "results.parquet"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
pq.write_table(results, RESULT_PATH, compression="zstd")
frame = results.to_pandas()
RESULT_PATH

In [ ]:
frame["weighted_correct"] = frame["correct"] * frame["paper_weight"]
frame["weighted_missing"] = frame["verdict"].isna() * frame["paper_weight"]
paper_summary = (
    frame.groupby("ctx_size", as_index=False)
    .agg(
        correct=("weighted_correct", "sum"),
        total=("paper_weight", "sum"),
        no_verdict=("weighted_missing", "sum"),
    )
    .sort_values("ctx_size")
)
paper_summary["accuracy"] = paper_summary["correct"] / paper_summary["total"]
assert paper_summary["ctx_size"].tolist() == list(LENGTHS)
assert paper_summary["total"].tolist() == [600] * len(LENGTHS)
display(paper_summary)
plt.figure(figsize=(8, 4.5))
plt.plot(
    paper_summary["ctx_size"],
    paper_summary["accuracy"],
    marker="o",
)
plt.xticks(LENGTHS)
plt.ylim(0, 1)
plt.xlabel("Input length (# nominal tokens)")
plt.ylabel("Accuracy")
plt.title("FLenQA accuracy by input length — paper weighting")
plt.grid(alpha=0.25)
plt.show()

In [ ]:
unique_summary = (
    frame.groupby("ctx_size", as_index=False)
    .agg(
        correct=("correct", "sum"),
        total=("prompt_id", "size"),
        no_verdict=("verdict", lambda values: values.isna().sum()),
    )
    .sort_values("ctx_size")
)
unique_summary["accuracy"] = unique_summary["correct"] / unique_summary["total"]
assert (
    dict(zip(unique_summary["ctx_size"], unique_summary["total"], strict=True))
    == EXPECTED_UNIQUE_COUNTS
)
display(unique_summary)
plt.figure(figsize=(8, 4.5))
plt.plot(
    unique_summary["ctx_size"],
    unique_summary["accuracy"],
    marker="o",
)
plt.xticks(LENGTHS)
plt.ylim(0, 1)
plt.xlabel("Input length (# nominal tokens)")
plt.ylabel("Accuracy")
plt.title("FLenQA accuracy by input length — unique prompts")
plt.grid(alpha=0.25)
plt.show()

In [ ]:
task_summary = (
    frame.groupby(["task", "ctx_size"], as_index=False)
    .agg(correct=("correct", "sum"), total=("prompt_id", "size"))
    .sort_values(["task", "ctx_size"])
)
task_summary["accuracy"] = task_summary["correct"] / task_summary["total"]
display(task_summary)
plt.figure(figsize=(8, 4.5))
for task, task_frame in task_summary.groupby("task", sort=True):
    plt.plot(
        task_frame["ctx_size"],
        task_frame["accuracy"],
        marker="o",
        label=task,
    )
plt.xticks(LENGTHS)
plt.ylim(0, 1)
plt.xlabel("Input length (# nominal tokens)")
plt.ylabel("Accuracy")
plt.title("FLenQA unique-prompt accuracy by task")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

In [ ]:
verdict_labels = (
    frame["verdict"].map({True: "True", False: "False"}).fillna("No verdict")
)
verdict_counts = pd.crosstab(frame["ctx_size"], verdict_labels).reindex(
    index=LENGTHS, columns=["True", "False", "No verdict"], fill_value=0
)
token_lengths = (
    frame.groupby("ctx_size")["n_input_tokens"]
    .agg(["min", "median", "max"])
    .reindex(LENGTHS)
)
display("Verdict counts by nominal length", verdict_counts)
display("Exact Qwen token lengths by nominal bucket", token_lengths)

In [ ]:
upload_artifacts()
print("COLAB_NOTEBOOK_UPLOAD_COMPLETE")